In [ ]:
# allocations (you can edit these)
allocations = {
    "obsidian cutlery",
    "pyroflex cells",
    "thermalite core",
    "lava cake",
    "magma ink",
    "scoria paste",
    "ashes of the phoenix",
    "volcanic incense",
    "sulfur reactor",
}

# movements (you will fill these in)
movement = {
    "obsidian cutlery",
    "pyroflex cells",
    "thermalite core",
    "lava cake",
    "magma ink",
    "scoria paste",
    "ashes of the phoenix",
    "volcanic incense",
    "sulfur reactor",
}

def compute_profit(allocations, movement):
    profit = 1_000_000
    for product in allocations:
        x = allocations[product]
        # fee term
        profit -= 1_000_000 * (x ** 2)
        # return term
        profit += 1_000_000 * x * movement[product]
    return profit

if __name__ == "__main__":
    profit = compute_profit(allocations, movement)
    print("Total money:", profit)

Total money: 1000000


: 

In [1]:
import numpy as np
from scipy.optimize import minimize, Bounds, LinearConstraint

products = [
    "obsidian cutlery",
    "pyroflex cells",
    "thermalite core",
    "lava cake",
    "magma ink",
    "scoria paste",
    "ashes of the phoenix",
    "volcanic incense",
    "sulfur reactor",
]

def optimize_allocation(mvmt: np.ndarray) -> np.ndarray:
    """
    Optimize portfolio allocation given a movement vector.

    Maximizes: 1_000_000 - x @ x + x @ mvmt
    Subject to:
        - Each x[i] in [-1, 1]
        - sum(x) <= 1

    Uses scipy SLSQP (no cvxpy dependency).

    Parameters
    ----------
    mvmt : np.ndarray
        Predicted movement vector of shape (n,).

    Returns
    -------
    np.ndarray
        Optimal allocation vector x of shape (n,).
    """
    n = mvmt.shape[0]

    # Minimize x'x - mvmt'x  (equivalent to maximising the profit; constant drops out)
    def objective(x):
        return x @ x - mvmt @ x

    def gradient(x):
        return 2 * x - mvmt

    # Per-element bounds: x[i] in [-1, 1]
    bounds = Bounds(lb=-1.0, ub=1.0)

    # Linear constraint: sum(x) <= 1
    sum_constraint = LinearConstraint(np.ones((1, n)), lb=-np.inf, ub=1.0)

    # Warm-start: unconstrained optimum clipped to bounds
    x0 = np.clip(mvmt / 2, -1.0, 1.0)

    result = minimize(
        objective,
        x0,
        jac=gradient,
        method="SLSQP",
        bounds=bounds,
        constraints=sum_constraint,
    )

    if not result.success:
        raise RuntimeError(f"Optimisation failed: {result.message}")

    return result.x

In [8]:
# ---------------------------------------------------------------------------
# Demo
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    mvmt = np.array([0.04, 0.08, -0.14, -0.40, 0.40, 0.00, 0.00, 0.02, 0.06])
    print("Movement vector:   ", mvmt)

    x_opt = optimize_allocation(mvmt)
    profit = 1_000_000 * (1 - x_opt @ x_opt + x_opt @ mvmt)

    print("Optimal allocation:", np.round(x_opt, 6))
    print(f"Sum of allocations:  {x_opt.sum():.6f}  (must be <= 1)")
    print(f"All in [-1, 1]:      {np.all(x_opt >= -1 - 1e-9) and np.all(x_opt <= 1 + 1e-9)}")
    print(f"Optimal profit:      {profit:,.4f}")

Movement vector:    [ 0.04  0.08 -0.14 -0.4   0.4   0.    0.    0.02  0.06]
Optimal allocation: [ 0.02  0.04 -0.07 -0.2   0.2   0.    0.    0.01  0.03]
Sum of allocations:  0.030000  (must be <= 1)
All in [-1, 1]:      True
Optimal profit:      1,087,900.0000


In [3]:
# ---------------------------------------------------------------------------
# Demo
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    mvmt = np.array([0, 0.2, -0.15, -0.1, 0.5, 0, 0, 0.05, 0])
    print("Movement vector:   ", mvmt)

    x_opt = optimize_allocation(mvmt)
    profit = 1_000_000 * (1 - x_opt @ x_opt + x_opt @ mvmt)

    print("Optimal allocation:", np.round(x_opt, 6))
    print(f"Sum of allocations:  {x_opt.sum():.6f}  (must be <= 1)")
    print(f"All in [-1, 1]:      {np.all(x_opt >= -1 - 1e-9) and np.all(x_opt <= 1 + 1e-9)}")
    print(f"Optimal profit:      {profit:,.4f}")

Movement vector:    [ 0.    0.2  -0.15 -0.1   0.5   0.    0.    0.05  0.  ]
Optimal allocation: [ 0.     0.1   -0.075 -0.05   0.25   0.     0.     0.025  0.   ]
Sum of allocations:  0.250000  (must be <= 1)
All in [-1, 1]:      True
Optimal profit:      1,081,250.0000
